# Topic Modeling with BERTopic — Parameter Search & Evaluation (iGEM Teams)

Loads the pre-computed **iGEM Teams** embeddings, runs a grid search over
key UMAP/HDBSCAN parameters, evaluates each configuration with **C_v
coherence**, **topic diversity**, and **DBCV**, selects the best model,
optionally reassigns outliers, and saves the results to `assets/topic_models/`.

> **Recommended** — this is the notebook used in the associated publication.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 02-topic_model/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import MODELS_DIR, set_seed
from aux.topic_modeling import load_corpus, save_topic_outputs
from aux.evaluation import grid_search

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
EMBEDDINGS_FILE = "teams_embeddings.npy"
CORPUS_FILE     = "teams_corpus.txt"
ID_COL          = "UT"
PREFIX          = "teams"

PARAM_GRID = {
    "min_cluster_size": [8, 10, 15, 20],
    "umap_n_neighbors": [10, 15, 25],
    "umap_n_components": [5, 10],
}

# Reassign all iGEM noise documents to their nearest topic (see section 3).
REDUCE_OUTLIERS = True

## 1. Load embeddings and corpus

In [3]:
embeddings, corpus = load_corpus(EMBEDDINGS_FILE, CORPUS_FILE)
docs = corpus["text"].tolist()
print(f"Teams: {embeddings.shape[0]:,} docs, {embeddings.shape[1]} dims")

Teams: 4,707 docs, 384 dims


## 2. Grid search

Fit and evaluate a BERTopic model for every parameter combination.

In [4]:
results, best = grid_search(docs, embeddings, PARAM_GRID, label="Teams")
results

[Teams] 1/24  mcs=8, nn=10, nc=5 ... topics=191, C_v=0.5786, div=0.7508, DBCV=0.2269
[Teams] 2/24  mcs=8, nn=10, nc=10 ... topics=194, C_v=0.5644, div=0.7608, DBCV=0.2448
[Teams] 3/24  mcs=8, nn=15, nc=5 ... topics=194, C_v=0.5551, div=0.7387, DBCV=0.2202
[Teams] 4/24  mcs=8, nn=15, nc=10 ... topics=176, C_v=0.5643, div=0.7483, DBCV=0.2299
[Teams] 5/24  mcs=8, nn=25, nc=5 ... topics=161, C_v=0.5974, div=0.7143, DBCV=0.1550
[Teams] 6/24  mcs=8, nn=25, nc=10 ... topics=164, C_v=0.5682, div=0.7396, DBCV=0.1868
[Teams] 7/24  mcs=10, nn=10, nc=5 ... topics=150, C_v=0.5819, div=0.6913, DBCV=0.2363
[Teams] 8/24  mcs=10, nn=10, nc=10 ... topics=151, C_v=0.5665, div=0.7146, DBCV=0.2077
[Teams] 9/24  mcs=10, nn=15, nc=5 ... topics=153, C_v=0.5639, div=0.7007, DBCV=0.2142
[Teams] 10/24  mcs=10, nn=15, nc=10 ... topics=137, C_v=0.5739, div=0.6737, DBCV=0.1992
[Teams] 11/24  mcs=10, nn=25, nc=5 ... topics=129, C_v=0.5833, div=0.6519, DBCV=0.1894
[Teams] 12/24  mcs=10, nn=25, nc=10 ... topics=124, C

,min_cluster_size,n_neighbors,n_components,n_topics,outlier_frac,coherence_cv,diversity,dbcv
0,8,25,5,161,0.2462,0.5974,0.7143,0.1550
1,10,25,5,129,0.2379,0.5833,0.6519,0.1894
2,10,10,5,150,0.2205,0.5819,0.6913,0.2363
3,8,10,5,191,0.2071,0.5786,0.7508,0.2269
4,10,25,10,124,0.2365,0.5747,0.6540,0.1410
5,10,15,10,137,0.2003,0.5739,0.6737,0.1992
6,8,25,10,164,0.2558,0.5682,0.7396,0.1868
7,10,10,10,151,0.2141,0.5665,0.7146,0.2077
8,8,10,10,194,0.2101,0.5644,0.7608,0.2448
9,8,15,10,176,0.1997,0.5643,0.7483,0.2299


In [5]:
print("Best configuration:")
for k in ["min_cluster_size", "n_neighbors", "n_components", "n_topics",
          "coherence_cv", "diversity", "dbcv", "outlier_frac"]:
    print(f"  {k:16s} = {best[k]}")

Best configuration:
  min_cluster_size = 8
  n_neighbors      = 25
  n_components     = 5
  n_topics         = 161
  coherence_cv     = 0.5974
  diversity        = 0.7143
  dbcv             = 0.155
  outlier_frac     = 0.2462


## 3. Reduce outliers (optional)

HDBSCAN labels documents that fall outside any dense cluster as topic **−1**
(noise). While that is acceptable for the SynBio literature (some papers may be
genuinely off-topic), every iGEM team project is by definition related to
synthetic biology — its text may simply be too short or idiosyncratic to land in
a cluster. BERTopic's `reduce_outliers` (strategy `"embeddings"`, threshold `0`)
reassigns **all** noise documents to their nearest topic by cosine similarity,
without retraining the model.

Controlled by `REDUCE_OUTLIERS` in the config above (enabled for teams,
disabled for papers).

In [6]:
model = best["model"]
topics = list(best["topics"])

if REDUCE_OUTLIERS:
    before = sum(1 for t in topics if t == -1)
    topics = model.reduce_outliers(
        docs, topics, strategy="embeddings", embeddings=embeddings, threshold=0,
    )
    model.update_topics(docs, topics=topics)
    after = sum(1 for t in topics if t == -1)
    print(f"Outliers: {before:,} → {after:,}")
else:
    print("Outlier reduction disabled — keeping HDBSCAN noise labels.")

2026-06-02 10:11:21,951 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers: 1,159 → 0


## 4. Save best model, outputs, and grid-search results

In [7]:
save_topic_outputs(model, corpus, topics, ID_COL, PREFIX)
results.to_csv(MODELS_DIR / f"{PREFIX}_grid_search.txt", sep="\t", index=False)

print(f"Saved → {MODELS_DIR}")
for f in sorted(MODELS_DIR.glob(f"{PREFIX}_*")):
    print(f"  {f.name}")

2026-06-02 10:11:24,964 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models
  teams_doc_topics.txt
  teams_grid_search.txt
  teams_topic_info.txt
  teams_topic_model
  teams_topic_names.txt
